In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.script.variable_profiling import eda_per_table_printing_results
from default_risk.script.variable_profiling import eda_per_table_persisting_result_html
from default_risk.script.variable_profiling import create_files_nulls_per_colmun
from default_risk.script.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.script.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids

import default_risk.config as cfg
import dtale
import logging


installments_payment_df = pd.read_csv(cfg.INSTALLMENTS_PAYMENTS)

column_order_reference="DAYS_INSTALMENT"

data_frame_size=len(installments_payment_df)


installments_payment_df.sort_values(["SK_ID_PREV",column_order_reference,"DAYS_ENTRY_PAYMENT"],inplace=True)


log = logging.getLogger('werkzeug')


with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)




Invariants:

Soft constraints:

Decisions summary: 

1- We will divide the missing values  at AMT_PAYMENT -  DAYS_ENTRY_PAYMENT in 2 diferent cases at the moment of taking metrics.
    a- Dead tails: Where the values miss and stay as nan in the remaining rows of the serie and show now relevant change in other variable. This suggest administral padding 
    so we will ignore dead tails to the compute of agg metrics.

    b- Remaining cases: Are less than 50, even if are interesting we don't have enough observations to modelate it, so we will input it with foward

In [ ]:
#files for the data dictionary
create_files_nulls_per_colmun(installments_payment_df,"installments_payment")

In [ ]:
#runing screening script
eda_per_table_printing_results(installments_payment_df,schema,"installments_payment",False)

In [ ]:
prev_id= installments_payment_df["SK_ID_PREV"].unique()
series_to_show= prev_id[:100]
dtale.show(recreate_and_sort_the_serie_given_ids(series_to_show,installments_payment_df,column_order_reference))

In [ ]:
#in order to undestand the nature behind the missing values of DAYS_ENTRY_PAYMENT
rows_with_nulls=installments_payment_df[installments_payment_df["DAYS_ENTRY_PAYMENT"].isna()]
dtale.show(recreate_and_sort_series_given_rows(rows_with_nulls,installments_payment_df,column_order_reference))

#in all the visualizated cases the tendence of the missing values is to show ups at the end of the secuence, like padding, so a "dead tail" definition is needed.
#Also, in the contract of SK_ID_PREV= 1004174 we can observe a jump from 8 to 101 in "NUM_INSTALMENT_NUMBER".
#This suggest that could exist errors in the counter or a sentinel / special value (101 show ups in more rows).

In [ ]:
#now, let's validate the hipotesis of missing values in DAYS_ENTRY_PAYMENT y AMT_PAYMENT are dead tails. For that we will analize if exists cases
#that once the serie have a missing value in these columns, can exist a row with a value different a nan, of if once hit nan, it's always nan without relevant changes.
installments_payment_df["NEXT_VALUE_PAYMENT"] = installments_payment_df.groupby("SK_ID_PREV")["DAYS_ENTRY_PAYMENT"].shift(-1)
not_a_deadtail_mask= (installments_payment_df["DAYS_ENTRY_PAYMENT"].isna() ) & ( installments_payment_df["NEXT_VALUE_PAYMENT"].notna())
print(not_a_deadtail_mask.sum())
potencial_incosistencies_rows= installments_payment_df[not_a_deadtail_mask]
dtale.show(recreate_and_sort_series_given_rows(potencial_incosistencies_rows,installments_payment_df,column_order_reference))
#this series exhib an anormal behaivor. Seems like have another schedule with their own counter, using the prefix "100" (101,102,103...) in NUM_INSTALLMENT_NUMBER and a different NUM_INSTALLMEMT_VERSION.
#Anyways when we exclude the dead tail cases the remaining observation are less than 50. 


44


In [ ]:
#now let's analize the behaivor of the rows when the installment number jump to 100.
groups_sizes= installments_payment_df.groupby("SK_ID_PREV")["SK_ID_PREV"].transform("size")
series_bellow_hundred_rows= installments_payment_df[groups_sizes < 100]
rows_with_notation= series_bellow_hundred_rows[series_bellow_hundred_rows["NUM_INSTALMENT_NUMBER"] > 100]
dtale.show(recreate_and_sort_series_given_rows(rows_with_notation,installments_payment_df,"NUM_INSTALMENT_NUMBER")) 
